# Lab 5: Automate the Monthly Report
**BUS-G 492 · Coding with AI for Business Analysts**

In [ ]:
# --- Setup: run this cell first -------------------------------------------
# Finds the course data whether you are in Colab or on your own laptop.
# INSTRUCTOR: set DATA_URL once to the raw GitHub folder that holds /data (see instructor README).
import os
DATA_URL = "https://raw.githubusercontent.com/ilovemypinktutu/g492-coding/main/data/"
for candidate in ["data/", "../data/", "../../data/"]:
    if os.path.exists(candidate + "orders.csv"):
        DATA = candidate
        break
else:
    DATA = DATA_URL
print("Reading data from:", DATA)

**Scenario.** Every month, someone at Limestone Outfitters spends a day building the same Excel sales report by hand: they download exchange rates, copy data, build pivots and format the tables. Your job is to turn that day into **one function call**.

By the end you will have:
1. pulled live data from a web **API**,
2. wrapped your analysis in a **function** with a parameter (`month`),
3. produced a formatted **Excel report** a manager would actually open,
4. generated six months of reports in a **loop**, and checked that they reconcile.

In [ ]:
import pandas as pd, requests, os
orders = pd.read_csv(DATA + "orders.csv", parse_dates=["order_date"])
items = pd.read_csv(DATA + "order_items.csv")
products = pd.read_csv(DATA + "products.csv")
customers = pd.read_csv(DATA + "customers.csv")
returns = pd.read_csv(DATA + "returns.csv", parse_dates=["return_date"])

## Part 1: Call an API (10 min)
An API is a URL that returns **data** instead of a web page. The European Central Bank publishes daily exchange rates. Frankfurter serves them for free, with no key needed.

Open this in a new browser tab first and look at what comes back:
`https://api.frankfurter.dev/v1/2026-03-01..2026-03-31?base=USD&symbols=EUR,GBP`

That format is **JSON**: nested `{key: value}` pairs. Python reads it into a dictionary.

In [ ]:
url = "https://api.frankfurter.dev/v1/2026-03-01..2026-03-31"
resp = requests.get(url, params={"base": "USD", "symbols": "EUR,GBP"}, timeout=20)
print(resp.status_code)          # 200 = OK
data = resp.json()
print(list(data.keys()))
list(data["rates"].items())[:3]

**1a.** Write a function `fx_for_month(month)` that takes `"2026-03"` and returns a dict like `{"USD": 1.0, "EUR": 1.08, "GBP": 1.27}`: **USD per 1 unit** of each currency, averaged over the month.

Notice that the API gives *EUR per USD*, and you need the reverse. It is also good practice to fall back to the saved file `fx_rates_monthly.csv` if the API is down, because automation that breaks when a website hiccups isn't automation.

In [ ]:
def fx_for_month(month):
    # your code (or AI's code, which you have read line by line)
    ...
fx_for_month("2026-03")

## Part 2: From analysis to a function (15 min)
**2a.** Write `monthly_kpis(month)` that returns a dict of DataFrames:
- `summary`: one column of KPIs: net revenue (USD), completed orders, average order value (USD), units sold, return rate (units returned ÷ units sold for this month's orders), # new customers (first order this month)
- `by_category`: net revenue USD and units by category, sorted
- `by_country`: net revenue USD and orders by country, sorted

Rules (same company definitions as Lab 1): completed orders only, no test accounts, net revenue = qty × price × (1 − discount), converted with `fx_for_month`.

**Tip:** get it working for one month *in plain cells* first, then indent it into a function. Ask AI to help, but give it these rules.

In [ ]:
def monthly_kpis(month):
    ...
k = monthly_kpis("2026-03")
k["summary"]

## Part 3: Write an Excel report someone would open (15 min)
**3a.** Write `build_report(month, folder="reports")` that saves `reports/sales_2026-03.xlsx` with three sheets: *Summary*, *By Category*, *By Country*. Make it look finished:
- bold header row, sensible column widths
- currency format `$#,##0` for money, `0.0%` for rates
- a bar chart of revenue by category on the *By Category* sheet
- a note at the bottom of *Summary*: data source, FX source, and when the report was generated

`openpyxl` does all of this. The AI knows it well, but check every number format it chooses.

In [ ]:
def build_report(month, folder="reports"):
    ...
build_report("2026-03")

## Part 4: Run it for six months, and prove it reconciles (10 min)
**4a.** Loop over `2026-01` … `2026-06` and build all six reports.

**4b. Reconciliation check.** Recompute H1 2026 net revenue **a different way**: one big query over all six months at once, using `fx_rates_monthly.csv` instead of the API. The sum of your six reports must match within 0.5%. (Why might it not match *exactly*? Write your answer.)

In [ ]:
# 4a
# 4b

## Part 5: Make it run without you (discussion + homework)
A function is only half of automation. Something still has to **call** it on the first of every month. Options, from least to most engineering:

| Option | How | Good for |
|---|---|---|
| Colab / Jupyter scheduled run | Colab Pro scheduling or a laptop cron job | personal use |
| **GitHub Actions** | a YAML file that runs your script on a cron schedule and emails or uploads the file | small teams, free |
| Power Automate / Zapier | trigger → run a script → post to Teams/Slack/SharePoint | Microsoft-shop companies |
| Airflow / dbt / cloud schedulers | pipelines owned by a data team | production |

**Homework:** ask an AI to write a GitHub Actions workflow that runs `build_report` for the *previous month* at 7am on the 1st of every month. Before you trust it, explain **every line** of the YAML in a comment, and name one way it could fail silently.